# Expand genera in a samplesheet using GTDB R226

**This notebook is NOT a Nextflow pipeline module.** It runs standalone, before
`nextflow run`, to rewrite a samplesheet so that genus-only rows are replaced
with N species rows drawn from the GTDB Release 226.0 (RS226) taxonomy.

```
samplesheet.csv  ──►  THIS NOTEBOOK  ──►  samplesheet.expanded.csv  ──►  nextflow run ...
```

For 2–3 specific genera in a one-off analysis. The default pipeline behavior
(one arbitrary species per genus) stays untouched — this just produces a
different samplesheet.

**Method summary** (also printed in Cell 10 as a ready-to-paste paragraph):

1. Download GTDB R226 bacterial taxonomy (cached at `~/.cache/mhcrefseq/gtdb/`).
2. For each target genus, enumerate all species in GTDB.
3. Filter to species that have a UniProt reference proteome (mirrors the
   pipeline's own query so positives here will succeed in `download_fasta.sh`).
4. Draw a seeded random sample of up to `MAX_PER_GENUS` species per genus —
   avoids alphabetical / quality bias of a sort-then-truncate.
5. Rewrite the samplesheet, write an audit TSV, print the methods paragraph.

Cell asserts catch silent bugs at each transformation boundary.


## 1. Imports & config

In [ ]:
import csv, gzip, hashlib, random, re, time, urllib.parse, urllib.request
from pathlib import Path
import pandas as pd

# --- GTDB release pinning ---
GTDB_RELEASE = "226.0"
GTDB_URL = (
    f"https://data.gtdb.ecogenomic.org/releases/release226/{GTDB_RELEASE}"
    f"/bac120_taxonomy_r226.tsv.gz"
)
CACHE_DIR  = Path.home() / ".cache" / "mhcrefseq" / "gtdb"
CACHE_PATH = CACHE_DIR / "bac120_taxonomy_r226.tsv.gz"

# --- Per-analysis inputs (edit these) ---
SAMPLESHEET_IN   = Path("samplesheet.csv")
SAMPLESHEET_OUT  = Path("samplesheet.expanded.csv")
AUDIT_LOG        = Path("expansion_audit.tsv")
GENERA_TO_EXPAND = ["Escherichia", "Listeria", "Bacillus"]
MAX_PER_GENUS    = 10        # cap on randomly-selected species per genus
RANDOM_SEED      = 42        # reruns are bit-identical with same seed

# --- Behavior toggles ---
COLLAPSE_GTDB_LINEAGES = True   # treat "Escherichia coli_A" as "Escherichia coli"
UNIPROT_THROTTLE_SEC   = 0.1    # sleep between UniProt REST calls

# --- Asserts on config ---
assert isinstance(GENERA_TO_EXPAND, list) and len(GENERA_TO_EXPAND) >= 1
assert all(isinstance(g, str) and " " not in g for g in GENERA_TO_EXPAND), (
    f"GENERA_TO_EXPAND must contain single-token genus names; got {GENERA_TO_EXPAND}"
)
assert MAX_PER_GENUS >= 1
assert isinstance(RANDOM_SEED, int)

print(f"GTDB release      : {GTDB_RELEASE}")
print(f"Genera to expand  : {GENERA_TO_EXPAND}")
print(f"Max per genus     : {MAX_PER_GENUS}")
print(f"Random seed       : {RANDOM_SEED}")
print(f"Collapse lineages : {COLLAPSE_GTDB_LINEAGES}")
print(f"Input samplesheet : {SAMPLESHEET_IN}")
print(f"Output samplesheet: {SAMPLESHEET_OUT}")
print(f"Audit log         : {AUDIT_LOG}")


## 2. Download GTDB taxonomy (cached)

First run downloads ~7 MB to `~/.cache/mhcrefseq/gtdb/`. Subsequent runs use the
cached file. SHA256 is computed and recorded in the audit log so you can detect
upstream changes if the file gets re-downloaded later.

In [ ]:
CACHE_DIR.mkdir(parents=True, exist_ok=True)

if CACHE_PATH.exists():
    size_mb = CACHE_PATH.stat().st_size / 1024**2
    print(f"Cache hit: {CACHE_PATH} ({size_mb:.1f} MB)")
else:
    print(f"Cache miss: downloading {GTDB_URL}")
    t0 = time.time()
    urllib.request.urlretrieve(GTDB_URL, CACHE_PATH)
    elapsed = time.time() - t0
    size_mb = CACHE_PATH.stat().st_size / 1024**2
    print(f"Downloaded {size_mb:.1f} MB in {elapsed:.1f}s")

# --- Asserts ---
assert CACHE_PATH.exists()
assert CACHE_PATH.stat().st_size > 1_000_000, "GTDB file unexpectedly small"
with open(CACHE_PATH, "rb") as f:
    magic = f.read(2)
assert magic == b"\x1f\x8b", "GTDB file is not a valid gzip"

h = hashlib.sha256()
with open(CACHE_PATH, "rb") as f:
    for chunk in iter(lambda: f.read(8192), b""):
        h.update(chunk)
GTDB_SHA256 = h.hexdigest()
print(f"SHA256: {GTDB_SHA256}")


## 3. Parse GTDB into a DataFrame

GTDB taxonomy strings look like:
`d__Bacteria;p__Proteobacteria;c__...;g__Escherichia;s__Escherichia coli`.

GTDB sometimes splits a Linnaean species into sub-clusters (e.g.
`Escherichia coli_A`, `Escherichia coli_B`). With `COLLAPSE_GTDB_LINEAGES = True`
we strip the trailing `_<UPPER>` so these are treated as one species for
selection purposes — UniProt reference proteomes are keyed by Linnaean species
anyway, so the sub-cluster suffix would not survive downstream.

In [ ]:
RE_GENUS   = re.compile(r"g__([^;]+)")
RE_SPECIES = re.compile(r"s__([^;]+)")
RE_LINEAGE_SUFFIX = re.compile(r"_[A-Z]+$")

rows = []
with gzip.open(CACHE_PATH, "rt", encoding="utf-8") as f:
    reader = csv.reader(f, delimiter="\t")
    for record in reader:
        if len(record) < 2:
            continue
        genome_id, tax = record[0], record[1]
        g = RE_GENUS.search(tax)
        s = RE_SPECIES.search(tax)
        if not g or not s:
            continue
        genus = g.group(1).strip()
        species_raw = s.group(1).strip()
        if not genus or not species_raw:
            continue
        rows.append((genome_id, genus, species_raw))

gtdb = pd.DataFrame(rows, columns=["genome_id", "genus", "species_raw"])

def _collapse(name: str) -> str:
    parts = name.rsplit(" ", 1)
    if len(parts) == 2:
        last_clean = RE_LINEAGE_SUFFIX.sub("", parts[1])
        return f"{parts[0]} {last_clean}"
    return RE_LINEAGE_SUFFIX.sub("", name)

gtdb["species_collapsed"] = gtdb["species_raw"].map(_collapse)
gtdb["species"] = gtdb["species_collapsed"] if COLLAPSE_GTDB_LINEAGES else gtdb["species_raw"]

print(f"Genomes                    : {len(gtdb):,}")
print(f"Unique genera              : {gtdb['genus'].nunique():,}")
print(f"Unique species (raw)       : {gtdb['species_raw'].nunique():,}")
print(f"Unique species (collapsed) : {gtdb['species_collapsed'].nunique():,}")
print(f"Active 'species' column    : {'collapsed' if COLLAPSE_GTDB_LINEAGES else 'raw'}")

# --- Asserts ---
assert len(gtdb) > 700_000, f"Unexpectedly few genomes: {len(gtdb)}"
assert gtdb["genus"].notna().all()
assert gtdb["species"].notna().all()
assert (gtdb["genus"] != "").all()
assert gtdb["species_raw"].nunique() > 100_000


## 4. Load and inspect the input samplesheet

In [ ]:
assert SAMPLESHEET_IN.exists(), f"Input samplesheet not found: {SAMPLESHEET_IN}"

samples = pd.read_csv(SAMPLESHEET_IN)
print("Samplesheet shape:", samples.shape)
print()
print(samples.to_string(index=False))

# --- Asserts ---
assert list(samples.columns) == ["sample", "scientific_name"], (
    f"Unexpected columns (expected ['sample', 'scientific_name']): {list(samples.columns)}"
)
assert samples.notna().all().all(), "NaN values in samplesheet"
assert samples["sample"].astype(str).str.match(r"^[A-Za-z0-9_-]+$").all(), (
    "Sample IDs must contain only letters, digits, underscore, hyphen"
)

n_genus_rows = samples["scientific_name"].isin(GENERA_TO_EXPAND).sum()
print(f"\nRows matching GENERA_TO_EXPAND: {n_genus_rows} / {len(samples)}")
missing = [g for g in GENERA_TO_EXPAND if g not in samples["scientific_name"].values]
if missing:
    print(f"WARNING: these genera are in GENERA_TO_EXPAND but not in the samplesheet: {missing}")


## 5. Per-genus candidate species (full list, no cap)

Eyeball this output. The next cell hits UniProt for each candidate; if you
spot lineages you definitely don't want included, you can either change
`COLLAPSE_GTDB_LINEAGES` and rerun from Cell 3, or just adjust `MAX_PER_GENUS`
or the random seed and accept the seeded sample.

In [ ]:
candidates = {}  # genus -> sorted list of unique species
for genus in GENERA_TO_EXPAND:
    species_in_genus = sorted(gtdb.loc[gtdb["genus"] == genus, "species"].unique())
    candidates[genus] = species_in_genus
    print(f"\n=== {genus} ===")
    print(f"  {len(species_in_genus)} unique species in GTDB R{GTDB_RELEASE}")
    if not species_in_genus:
        print("  WARNING: no species found — check spelling against GTDB.")
        continue
    for sp in species_in_genus[:50]:
        print(f"    {sp}")
    if len(species_in_genus) > 50:
        print(f"    ... ({len(species_in_genus) - 50} more)")


## 6. UniProt reference-proteome cross-check

Mirrors the pipeline's own query in `bin/download_fasta.sh`:
`taxonomy_name:<species_with_+>+AND+proteome_type:reference`. A `True` here
means the pipeline will successfully retrieve *something* for this name.

Note: `taxonomy_name` is fuzzy. A query for `Escherichia somespecies` may
match the genus and return an arbitrary E. coli proteome. That mirrors the
pipeline's existing behavior — what you see is what you get downstream.

In [ ]:
UNIPROT_BASE = "https://rest.uniprot.org/proteomes/search"

def uniprot_has_reference_proteome(species: str):
    encoded = species.replace(" ", "+")
    url = (
        f"{UNIPROT_BASE}?query=taxonomy_name:{encoded}+AND+proteome_type:reference"
        f"&format=tsv&fields=upid&size=1"
    )
    try:
        with urllib.request.urlopen(url, timeout=30) as resp:
            status = resp.status
            body = resp.read().decode("utf-8")
    except Exception as e:
        print(f"  ERROR querying UniProt for '{species}': {e}")
        return (False, None, None)
    lines = [ln for ln in body.splitlines() if ln.strip()]
    if len(lines) < 2:
        return (False, None, status)
    upid = lines[1].split("\t")[0]
    return (True, upid, status)

records = []
for genus, species_list in candidates.items():
    print(f"\n=== {genus}: querying {len(species_list)} candidates ===")
    for sp in species_list:
        has, upid, status = uniprot_has_reference_proteome(sp)
        records.append({
            "genus": genus, "species": sp,
            "has_proteome": has, "upid": upid, "http_status": status,
        })
        time.sleep(UNIPROT_THROTTLE_SEC)
    n_yes = sum(1 for r in records if r["genus"] == genus and r["has_proteome"])
    print(f"  {n_yes} / {len(species_list)} have reference proteomes")

candidates_df = pd.DataFrame(records)

for genus in GENERA_TO_EXPAND:
    print(f"\n=== {genus} (annotated) ===")
    sub = candidates_df[candidates_df["genus"] == genus]
    print(sub.to_string(index=False))

# --- Asserts ---
assert len(candidates_df) == sum(len(v) for v in candidates.values())
for genus in GENERA_TO_EXPAND:
    n_yes = ((candidates_df["genus"] == genus) & candidates_df["has_proteome"]).sum()
    if candidates[genus] and n_yes == 0:
        print(f"WARNING: genus '{genus}' has 0 species with UniProt reference proteome")


## 7. Seeded random selection (the "no bias" cell)

Per-genus deterministic RNG: seed is `f"{RANDOM_SEED}:{genus}"`, so adding or
removing a genus from `GENERA_TO_EXPAND` does not perturb the picks for the
others. The seed and `MAX_PER_GENUS` are recorded in the audit log.

In [ ]:
SELECTED_SPECIES = {}
for genus in GENERA_TO_EXPAND:
    pool = sorted(
        candidates_df.query("genus == @genus and has_proteome")["species"].tolist()
    )
    rng = random.Random(f"{RANDOM_SEED}:{genus}")
    n = min(MAX_PER_GENUS, len(pool))
    pick = sorted(rng.sample(pool, n)) if pool else []
    SELECTED_SPECIES[genus] = pick
    print(f"\n=== {genus} ===")
    print(f"  Pool (UniProt-positive)            : {len(pool)}")
    print(f"  Selected (n=min(MAX_PER_GENUS, pool)): {len(pick)}")
    print(f"  RNG seed                           : '{RANDOM_SEED}:{genus}'")
    for sp in pick:
        print(f"    {sp}")

# --- Asserts ---
for genus in GENERA_TO_EXPAND:
    pool_size = ((candidates_df["genus"] == genus) & candidates_df["has_proteome"]).sum()
    assert len(SELECTED_SPECIES[genus]) == min(MAX_PER_GENUS, pool_size)
    assert len(SELECTED_SPECIES[genus]) == len(set(SELECTED_SPECIES[genus]))

# Idempotency check — second draw must equal first
_check = {}
for genus in GENERA_TO_EXPAND:
    pool = sorted(candidates_df.query("genus == @genus and has_proteome")["species"].tolist())
    rng = random.Random(f"{RANDOM_SEED}:{genus}")
    n = min(MAX_PER_GENUS, len(pool))
    _check[genus] = sorted(rng.sample(pool, n)) if pool else []
assert _check == SELECTED_SPECIES, "Random selection is not deterministic — seed bug"
print("\nIdempotency check: OK")


## 8. Build the expanded samplesheet

In [ ]:
expanded_rows = []
expansion_summary = {}

for _, row in samples.iterrows():
    sample = row["sample"]
    name = row["scientific_name"]
    if name in GENERA_TO_EXPAND:
        picks = SELECTED_SPECIES[name]
        for sp in picks:
            expanded_rows.append({"sample": sample, "scientific_name": sp})
        expansion_summary.setdefault(sample, []).append((name, len(picks)))
    else:
        expanded_rows.append({"sample": sample, "scientific_name": name})

expanded = pd.DataFrame(expanded_rows, columns=["sample", "scientific_name"])

print(f"Input  : {len(samples):4d} rows, {samples['sample'].nunique()} unique samples")
print(f"Output : {len(expanded):4d} rows, {expanded['sample'].nunique()} unique samples")
print()
print("Expansions per sample:")
if not expansion_summary:
    print("  (none — no rows matched GENERA_TO_EXPAND)")
for sample, ex in expansion_summary.items():
    for genus, n in ex:
        print(f"  {sample}: {genus} -> {n} species")

# --- Asserts ---
assert set(expanded["sample"]) == set(samples["sample"]), "Some samples lost during expansion"
for genus in GENERA_TO_EXPAND:
    bad = expanded[expanded["scientific_name"] == genus]
    assert bad.empty, f"Genus name '{genus}' still in expanded output:\n{bad}"
dupes = expanded[expanded.duplicated(subset=["sample", "scientific_name"])]
assert dupes.empty, f"Duplicate (sample, scientific_name) pairs:\n{dupes}"


## 9. Write expanded samplesheet + audit log

In [ ]:
expanded.to_csv(SAMPLESHEET_OUT, index=False)

audit_rows = []
for _, row in samples.iterrows():
    sample = row["sample"]
    name = row["scientific_name"]
    if name not in GENERA_TO_EXPAND:
        continue
    n_in_gtdb = int((gtdb["genus"] == name).sum())
    n_uniprot = int(((candidates_df["genus"] == name) & candidates_df["has_proteome"]).sum())
    picks = SELECTED_SPECIES[name]
    audit_rows.append({
        "sample": sample,
        "original_input": name,
        "gtdb_release": GTDB_RELEASE,
        "gtdb_sha256": GTDB_SHA256,
        "n_in_gtdb": n_in_gtdb,
        "n_with_uniprot_proteome": n_uniprot,
        "n_selected": len(picks),
        "max_per_genus": MAX_PER_GENUS,
        "random_seed": RANDOM_SEED,
        "selected_species": ";".join(picks),
    })
audit = pd.DataFrame(audit_rows)
audit.to_csv(AUDIT_LOG, sep="\t", index=False)

print(f"Wrote expanded samplesheet: {SAMPLESHEET_OUT.resolve()}")
print(f"Wrote audit log           : {AUDIT_LOG.resolve()}")
print(f"\nGTDB release : {GTDB_RELEASE}")
print(f"GTDB SHA256  : {GTDB_SHA256}")
print(f"Random seed  : {RANDOM_SEED}")

# --- Asserts ---
assert SAMPLESHEET_OUT.exists() and SAMPLESHEET_OUT.stat().st_size > 0
assert AUDIT_LOG.exists() and AUDIT_LOG.stat().st_size > 0


## 10. Methods-section snippet

In [ ]:
total_orig   = sum((samples["scientific_name"] == g).sum() for g in GENERA_TO_EXPAND)
total_picked = sum(len(SELECTED_SPECIES[g]) for g in GENERA_TO_EXPAND)

methods = f"""For samples annotated only at the genus level, a per-genus species panel was
constructed prior to running the nf-core/mhcrefseq pipeline. The Genome Taxonomy
Database release {GTDB_RELEASE} (RS226) was used as the source of bacterial
species concepts. For each target genus ({", ".join(GENERA_TO_EXPAND)}), all
species in the genus were enumerated and filtered to those with a UniProt
reference proteome (queried via the UniProt REST API with
`taxonomy_name:<species> AND proteome_type:reference`, mirroring the pipeline's
own download query). From the filtered pool, up to {MAX_PER_GENUS} species per
genus were drawn using a seeded random sample (Python `random.Random` with seed
`{RANDOM_SEED}:<genus>`) to avoid alphabetical or quality-of-annotation bias.
{total_picked} species were selected across {total_orig} original genus rows.
The expanded samplesheet was then passed unchanged to the nf-core/mhcrefseq
pipeline for proteome download, merging, and CD-HIT clustering."""

print(methods)
